# test run and cost estimation

runs the slr processing + pyciam pipeline on a small slice to check everything works
end-to-end, then uses the timings to estimate how much the full run would cost.

the test uses:
- 1 temperature scenario (tlim2.0) instead of 5
- 1 workflow (wf_1f) instead of 2
- 5 monte carlo samples instead of 1,000
- 50 coastal segments instead of the full 20,000
- 3 dask workers

In [1]:
# install pyciam from the inequality branch (must happen before any pyCIAM import)
# the pip version has different function signatures and variable names
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps',
    '/home/jovyan/inequality-dscim-coastal/pyciam'
])

# verify it installed from the right place
import importlib
importlib.invalidate_caches()

import pyCIAM
print(f"pyciam location: {pyCIAM.__file__}")

import inspect
from pyCIAM.run import get_refA
print(f"get_refA signature: {inspect.signature(get_refA)}")

pyciam location: /srv/conda/envs/notebook/lib/python3.14/site-packages/pyCIAM/__init__.py
get_refA signature: (fts_map, output_path, econ_input_path, slr_input_path, params, surge_input_path=None, mc_dim='quantile', storage_options={}, quantiles=[0.5], eps=1, diaz_inputs=False, **model_kwargs)


In [2]:
import time
import json
import numpy as np
import pandas as pd
import xarray as xr
from collections import OrderedDict
from itertools import product
from pathlib import Path
from cloudpathlib import AnyPath
from gcsfs import GCSFileSystem

from config import *

timings = {}

## parameters

In [3]:
T_SCENARIOS = ['tlim2.0']
T_WORKFLOWS = ['wf_1f']
T_N_SAMP_PER_WF = 5
T_N_SAMP = 5
T_N_SEGMENTS = 50
T_SEG_CHUNKSIZE = 1

T_DIR = DIR_SCRATCH / 'test-run-slr-inputs'
T_SLR = T_DIR / 'costtest-slr.zarr'
T_SLIIDERS_SEG = T_DIR / 'costtest-sliiders-seg.zarr'
T_REFA = T_DIR / 'costtest-refa.zarr'
T_TMP = T_DIR / 'costtest-pyciam-tmp.zarr'
T_INTERMEDIATE = T_DIR / 'costtest-pyciam-intermediate.zarr'
T_FINAL = T_DIR / 'costtest-pyciam-final.zarr'

F_N_SCENARIOS = 5
F_N_WORKFLOWS = 2
F_N_SAMPLES = 1000
F_N_WORKERS = 800

print('parameters set.')

parameters set.


## dask cluster

In [4]:
import os
from dask_gateway import Gateway
from distributed import wait
from distributed.diagnostics.plugin import UploadDirectory

img = os.environ.get('JUPYTER_IMAGE', None)
gateway = Gateway()

for c in gateway.list_clusters():
    try:
        gateway.stop_cluster(c.name)
    except:
        pass
time.sleep(10)

cluster = gateway.new_cluster(
    idle_timeout=3600,
    profile='micro',
    **(dict(worker_image=img, scheduler_image=img) if img else {})
)
client = cluster.get_client()
cluster.scale(3)

start = time.time()
deadline = start + 300
while len(client.scheduler_info()['workers']) < 1 and time.time() < deadline:
    elapsed = int(time.time() - start)
    n = len(client.scheduler_info()['workers'])
    print(f"\rwaiting for workers... {elapsed}s, workers: {n}    ", end='', flush=True)
    time.sleep(5)

n = len(client.scheduler_info()['workers'])
print(f"\rworkers ready: {n}                                    ")

if n == 0:
    print("no workers after 5 min. try running this cell again.")
else:
    # upload pyciam to workers (they can't access local filesystem)
    import pyCIAM
    client.register_plugin(
        UploadDirectory(
            str(Path(pyCIAM.__file__).parent),
            update_path=True,
            restart_workers=False,
        ),
        name='pyciam-upload',
    )
    client.run(
        lambda: __import__('subprocess').check_call(
            ['pip', 'install', '-q', 'cloudpathlib']
        )
    )
    result = client.submit(
        lambda: str(__import__('pyCIAM.run', fromlist=['get_refA']))
    ).result(timeout=60)
    print(f"pyciam + cloudpathlib installed on {n} workers")
    print(f"dashboard: {client.dashboard_link}")

cluster

workers ready: 3                                    
pyciam + cloudpathlib installed on 3 workers
dashboard: /services/dask-gateway/clusters/jhub.5cad4ed83d4b42b28d36e7fa1f61c6cb/status


In [5]:
# wrappers import pyciam inside the worker so the scheduler
# never needs to deserialize pyciam objects
def run_get_refA(grp, **kwargs):
    from pyCIAM.run import get_refA
    return get_refA(grp, **kwargs)

def run_calc_all_cases(grp, **kwargs):
    from pyCIAM.run import calc_all_cases
    return calc_all_cases(grp, **kwargs)

def run_optimize_case(*args, **kwargs):
    from pyCIAM.run import optimize_case
    return optimize_case(*args, **kwargs)

def ensure_packages():
    import pyCIAM
    client.register_plugin(
        UploadDirectory(
            str(Path(pyCIAM.__file__).parent),
            update_path=True,
            restart_workers=False,
        ),
        name='pyciam-upload',
    )
    client.run(
        lambda: __import__('subprocess').check_call(
            ['pip', 'install', '-q', 'cloudpathlib']
        )
    )

## slr processing

In [6]:
import pint_xarray

def open_and_convert_zarr(ds_path):
    out = xr.open_zarr(ds_path)
    out['sea_level_change'] = (
        out.sea_level_change.pint.quantify().pint.to('meters').pint.dequantify()
    )
    return out

def open_and_convert_nc(ds_path):
    _path = str(ds_path).replace('gs://', '/gcs/')
    out = xr.open_dataset(_path)
    out['sea_level_change'] = (
        out.sea_level_change.pint.quantify().pint.to('meters').pint.dequantify()
    )
    return out

In [7]:
t0 = time.time()

np.random.seed(11222023)
nsamps = 20000
low = np.arange(0, nsamps, step=nsamps / T_N_SAMP_PER_WF)
high = low + nsamps / T_N_SAMP_PER_WF
quants = np.random.randint(low=low, high=high, size=None) / nsamps

local_dfs, global_dfs = [], []
for tlim, wf in product(T_SCENARIOS, T_WORKFLOWS):
    print(f'processing {tlim}/{wf}...')
    lp = f'{DIR_SLR_AR6_GRIDDED_PUBLIC}/{wf}/{tlim}win0.25/total-workflow.zarr'
    df = open_and_convert_zarr(lp)
    df = (
        df.sel(years=slice(2020, 2100), drop=True)
        .sea_level_change.chunk(dict(samples=-1))
        .quantile(quants, dim='samples')
        .assign_coords({'workflow': wf, 'tlim': tlim})
        .expand_dims(['workflow', 'tlim'])
    )
    local_dfs.append(df)

    gp = DIR_SLR_AR6_RAW / wf / f'{tlim}win0.25' / 'total-workflow.nc'
    gdf = open_and_convert_nc(gp)
    gdf = (
        gdf.sel(years=slice(2020, 2100), drop=True)
        .sea_level_change.chunk(dict(samples=-1))
        .quantile(quants, dim='samples')
        .assign_coords({'workflow': wf, 'tlim': tlim})
        .expand_dims(['workflow', 'tlim'])
    )
    global_dfs.append(gdf)

timings['slr_local_global'] = time.time() - t0
print(f'done: {timings["slr_local_global"]:.1f}s')

processing tlim2.0/wf_1f...
done: 13.3s


In [8]:
t0 = time.time()

fs = GCSFileSystem(requester_pays=True)
mapping = fs.get_mapper(PATH_VLM_REQUESTER_PAYS)
np.random.seed(11222023)
low_v = np.arange(0, nsamps, step=nsamps / T_N_SAMP)
high_v = low_v + nsamps / T_N_SAMP
quants_v = np.random.randint(low=low_v, high=high_v, size=None) / nsamps
vlm_df = open_and_convert_zarr(mapping)
vlm_df = (
    vlm_df.sel(years=slice(2020, 2100), drop=True)
    .sea_level_change.chunk(dict(samples=-1))
    .quantile(quants_v, dim='samples')
    .rename({'quantile': 'samples'})
    .assign_coords({'samples': np.arange(1, T_N_SAMP + 1)})
    .to_dataset()
)

timings['slr_vlm'] = time.time() - t0
print(f'vlm: {timings["slr_vlm"]:.1f}s')

vlm: 0.4s


In [ ]:
t0 = time.time()

df_full = xr.combine_by_coords(local_dfs)
df_full = df_full.stack(samples=['workflow', 'quantile'])
df_full = df_full.reset_index('samples').drop_vars(['workflow', 'quantile'])
df_full['samples'] = np.arange(1, T_N_SAMP + 1)

global_ds = xr.combine_by_coords(global_dfs)
global_ds = global_ds.stack(samples=['workflow', 'quantile'])
global_ds = global_ds.reset_index('samples').drop_vars(['workflow', 'quantile'])
global_ds['samples'] = np.arange(1, T_N_SAMP + 1)
global_ds = global_ds.squeeze(drop=True).sea_level_change

vlm_df['samples'] = np.arange(1, T_N_SAMP + 1)

all_ds = xr.Dataset({
    'lsl_msl05': df_full.sea_level_change,
    'lsl_ncc_msl05': vlm_df.sea_level_change,
    'gsl_msl05': global_ds,
    'lon': vlm_df.lon,
    'lat': df_full.lat,
})
all_ds['lon'] = all_ds.lon.where(all_ds.lon != -180, 180)
all_ds = all_ds.stack(locations=['lat', 'lon'])

# load everything to memory first (1 scenario, 5 samples fits fine)
all_ds = all_ds.load()

reduce_dims = [d for d in ['tlim', 'samples', 'years'] if d in all_ds.dims]
valid = (
    all_ds[['lsl_msl05', 'lsl_ncc_msl05']]
    .sel(years=slice(2100))
    .notnull().all(reduce_dims)
    .to_array('tmp').all('tmp')
)
all_ds = all_ds.sel(locations=valid.where(valid, drop=True).locations)

rename_map = {}
if 'years' in all_ds.dims: rename_map['years'] = 'year'
if 'samples' in all_ds.dims: rename_map['samples'] = 'sample'
if 'locations' in all_ds.dims: rename_map['locations'] = 'site_id'
if 'tlim' in all_ds.dims: rename_map['tlim'] = 'scenario'
all_ds = all_ds.rename(rename_map)

n_sites = len(all_ds.site_id)
all_ds = all_ds.chunk({'site_id': -1, 'scenario': 1, 'year': -1, 'sample': T_N_SAMP})

lat_vals = all_ds.coords['lat'].values
lon_vals = all_ds.coords['lon'].values
all_ds = all_ds.drop_vars(['lat', 'lon'])
all_ds['site_id'] = np.arange(len(all_ds.site_id))
all_ds['lat'] = ('site_id', lat_vals)
all_ds['lon'] = ('site_id', lon_vals)

for v in all_ds.data_vars:
    all_ds[v].encoding.clear()
for k, v in all_ds.coords.items():
    v.encoding.clear()
    if v.dtype == object:
        all_ds[k] = v.astype("unicode")

all_ds.to_zarr(str(T_SLR), mode='w', zarr_format=2)

timings['slr_combine_save'] = time.time() - t0
print(f'combine + save: {timings["slr_combine_save"]:.1f}s, {n_sites} sites')

## pyciam

In [ ]:
from pyCIAM.constants import CASES, COSTTYPES
from pyCIAM.io import create_template_dataarray
from pyCIAM.run import calc_all_cases, get_refA, optimize_case
from pyCIAM.utils import (
    add_attrs_to_result,
    collapse_econ_inputs_to_seg,
    subset_econ_inputs,
)

params = pd.read_json(PATH_PARAMS)['values']
quantiles = np.arange(1, T_N_SAMP + 1)
econ_input_path = str(PATH_SLIIDERS)
slr_input_paths = [T_SLR]
slr_names = ['ar6']
surge_input_paths = {k: AnyPath(v) for k, v in PATHS_SURGE_LOOKUP.items()}

In [ ]:
# check if sliiders has the variables pyciam expects
# pyciam may expect K_2019/pop_2019 but sliiders v1.2 might have K_2014/pop_2014
sliiders_check = xr.open_zarr(econ_input_path, chunks=None)
sliiders_vars = list(sliiders_check.data_vars)
print(f"sliiders variables: {sliiders_vars[:15]}...")

# check for the K and pop variable names
k_var = [v for v in sliiders_vars if v.startswith('K_')]
pop_var = [v for v in sliiders_vars if v.startswith('pop_')]
print(f"K variables: {k_var}")
print(f"pop variables: {pop_var}")

# check what collapse_econ_inputs_to_seg expects
import inspect
src = inspect.getsource(collapse_econ_inputs_to_seg)
expected_vars = [w.strip("'\"") for w in src.split('[')[1].split(']')[0].split(',')]
print(f"collapse expects: {expected_vars}")

sliiders_check.close()

In [ ]:
t0 = time.time()

# if sliiders has K_2014 but pyciam expects K_2019, patch the zarr temporarily
sliiders_tmp = xr.open_zarr(econ_input_path, chunks=None)
needs_rename = 'K_2019' not in sliiders_tmp.data_vars and 'K_2014' in sliiders_tmp.data_vars

if needs_rename:
    print("patching sliiders: renaming K_2014->K_2019, pop_2014->pop_2019 if needed")
    rename_dict = {}
    if 'K_2014' in sliiders_tmp.data_vars:
        rename_dict['K_2014'] = 'K_2019'
    if 'pop_2014' in sliiders_tmp.data_vars:
        rename_dict['pop_2014'] = 'pop_2019'
    sliiders_patched = sliiders_tmp.rename(rename_dict)

    # save patched version temporarily
    patched_path = T_DIR / 'sliiders-patched.zarr'
    for v in sliiders_patched.data_vars:
        sliiders_patched[v].encoding.clear()
    for k, v in sliiders_patched.coords.items():
        v.encoding.clear()
    sliiders_patched.to_zarr(str(patched_path), mode='w', zarr_format=2)
    econ_input_path_for_collapse = str(patched_path)
    print(f"patched sliiders saved to {patched_path}")
else:
    econ_input_path_for_collapse = econ_input_path
    print("sliiders has expected variables, no patching needed")

sliiders_tmp.close()

collapse_econ_inputs_to_seg(
    econ_input_path_for_collapse, T_SLIIDERS_SEG,
    seg_var_subset=None, output_chunksize=100,
    storage_options=STORAGE_OPTIONS, seg_var=SEG_VAR,
)
timings['collapse_sliiders'] = time.time() - t0
print(f'collapse sliiders: {timings["collapse_sliiders"]:.1f}s')

In [ ]:
ciam_in_full = subset_econ_inputs(
    xr.open_zarr(econ_input_path, chunks=None, storage_options=STORAGE_OPTIONS),
    SEG_VAR, seg_var_subset=None,
)
total_seg_ir = len(ciam_in_full[SEG_VAR])

test_seg_ir = ciam_in_full[SEG_VAR].values[:T_N_SEGMENTS]
ciam_in = ciam_in_full.sel({SEG_VAR: test_seg_ir})
ciam_in = ciam_in.sel(year=np.concatenate((
    np.arange(2040, 2060), np.arange(2080, 2100)
)))
print(f'total seg_ir: {total_seg_ir}, test seg_ir: {len(test_seg_ir)}')

In [ ]:
slr_test = xr.open_zarr(str(T_SLR), chunks=None)
scenarios = slr_test.scenario.values
print(f"scenarios: {scenarios}")

coords = OrderedDict({
    'case': CASES, 'costtype': COSTTYPES,
    SEG_VAR: ciam_in[SEG_VAR].values, 'scenario': scenarios,
    'sample': quantiles,
    'year': np.arange(params.model_start, ciam_in.year.max().item() + 1),
    **{d: ciam_in[d].values for d in ['ssp', 'iam'] if d in ciam_in.dims},
})

chunk_spec = {SEG_VAR: 1, 'case': len(coords['case']) - 1}
resolved_chunks = {
    k: chunk_spec.get(k, len(v) if hasattr(v, '__len__') else v)
    for k, v in coords.items()
}

out_ds = create_template_dataarray(
    coords.keys(), coords, resolved_chunks
).to_dataset(name='costs')
out_ds['npv'] = out_ds.costs.isel(year=0, costtype=0, drop=True).astype('float64')
out_ds['optimal_case'] = out_ds.npv.isel(case=0, drop=True).astype('uint8')
out_ds = add_attrs_to_result(out_ds, SEG_VAR, mc_dim='sample')

out_ds.to_zarr(str(T_TMP), compute=False, mode='w', storage_options=STORAGE_OPTIONS, zarr_format=2)
print(f'template: {dict(out_ds.sizes)}')

In [ ]:
ensure_packages()
t0 = time.time()

segs = np.unique(ciam_in.seg)
seg_grps_refa = [segs[i:i + REFA_SEG_CHUNKSIZE] for i in range(0, len(segs), REFA_SEG_CHUNKSIZE)]
samps = np.arange(1, T_N_SAMP + 1)
grps_refa = list(product(seg_grps_refa, [samps]))
print(f'refa groups: {len(grps_refa)}')

refa_futs = client.map(
    run_get_refA, grps_refa,
    output_path=str(T_REFA),
    econ_input_path=str(T_SLIIDERS_SEG),
    slr_input_path=slr_input_paths[0],
    params=params, surge_input_path=surge_input_paths['seg'],
    mc_dim=MC_DIM, storage_options=STORAGE_OPTIONS,
    quantiles=quantiles, diaz_inputs=False, eps=1,
)
wait(refa_futs)

timings['refa'] = time.time() - t0
n_ok = sum(1 for f in refa_futs if f.status == 'finished')
n_err = sum(1 for f in refa_futs if f.status == 'error')
print(f'refa: {timings["refa"]:.1f}s, ok: {n_ok}, errors: {n_err}')

if n_err > 0:
    for f in refa_futs:
        if f.status == 'error':
            try: f.result()
            except Exception as e: print(f"error: {e}")
            break

In [ ]:
ensure_packages()
t0 = time.time()

groups = [
    ciam_in[SEG_VAR].isel({SEG_VAR: slice(i, i + T_SEG_CHUNKSIZE)}).values
    for i in np.arange(0, len(ciam_in[SEG_VAR]), T_SEG_CHUNKSIZE)
]

groups_ser = (
    pd.Series(groups).explode().reset_index()
    .rename(columns={'index': 'group_id', 0: SEG_VAR})
    .set_index(SEG_VAR).group_id
)

grps_ciam = list(product(groups, [samps]))
print(f'calc_all_cases groups: {len(grps_ciam)}')

ciam_futs = np.array(client.map(
    run_calc_all_cases, grps_ciam,
    params=params, econ_input_path=econ_input_path,
    slr_input_paths=slr_input_paths, slr_names=slr_names,
    output_path=T_TMP, refA_path=T_REFA,
    surge_input_path=surge_input_paths[SEG_VAR],
    seg_var=SEG_VAR, mc_dim=MC_DIM, quantiles=quantiles,
    storage_options=STORAGE_OPTIONS, diaz_inputs=False, check=False,
))
wait(ciam_futs)

timings['calc_all_cases'] = time.time() - t0
n_ok = sum(1 for f in ciam_futs if f.status == 'finished')
n_err = sum(1 for f in ciam_futs if f.status == 'error')
print(f'calc_all_cases: {timings["calc_all_cases"]:.1f}s, ok: {n_ok}, err: {n_err}')

if n_err > 0:
    for f in ciam_futs:
        if f.status == 'error':
            try: f.result()
            except Exception as e: print(f"error: {e}")
            break

In [ ]:
ensure_packages()
t0 = time.time()

sample_ids = {0: np.arange(1, T_N_SAMP + 1)}
n_opt_groups = 1

seg_adm_ser = pd.Series(ciam_in[SEG_VAR].values)
seg_adm_ser.index = ciam_in.seg.values
seg_grps_opt = seg_adm_ser.groupby(seg_adm_ser.index).apply(list)

precurser_futs = (
    seg_adm_ser.to_frame(SEG_VAR)
    .join(seg_grps_opt.rename('seg_group'))
)
precurser_futs.loc[:, 'samples'] = [np.arange(0, n_opt_groups)] * len(precurser_futs)
precurser_futs = (
    precurser_futs.explode('samples')
    .set_index([SEG_VAR, 'samples'])
    .seg_group.explode().to_frame()
    .join(groups_ser, on='seg_group')
    .groupby([SEG_VAR, 'samples'])
    .group_id.apply(set).apply(list)
)

opt_futs = precurser_futs.reset_index(drop=False).apply(
    lambda row: client.submit(
        run_optimize_case, row[SEG_VAR], *row.group_id,
        quantiles=sample_ids[row['samples']],
        econ_input_path=econ_input_path,
        output_path=str(T_TMP), seg_var=SEG_VAR,
        eps=1, check=False, storage_options=STORAGE_OPTIONS,
    ), axis=1,
)
wait(opt_futs)

timings['optimization'] = time.time() - t0
n_ok = sum(1 for f in opt_futs if f.status == 'finished')
n_err = sum(1 for f in opt_futs if f.status == 'error')
print(f'optimization: {timings["optimization"]:.1f}s, ok: {n_ok}, err: {n_err}')

if n_err > 0:
    for f in opt_futs:
        if f.status == 'error':
            try: f.result()
            except Exception as e: print(f"error: {e}")
            break

In [ ]:
t0 = time.time()

AGG_VAR = 'impact_region'

t = xr.open_zarr(str(T_TMP))
t = t.sel(case=OUTPUT_CASES, ssp=OUTPUT_SSPS, year=OUTPUT_YEARS)[['costs']]
for v in t.data_vars:
    t[v].encoding.clear()
for k, v in t.coords.items():
    v.encoding.clear()
    if v.dtype == object:
        t[k] = v.astype('unicode')
t.to_zarr(str(T_INTERMEDIATE), mode='w', zarr_format=2)

out = xr.open_zarr(str(T_INTERMEDIATE), chunks={'case': -1, SEG_VAR: 2})
out['costs'] = (
    out.costs.groupby(ciam_in[AGG_VAR]).sum().chunk({AGG_VAR: 2})
).persist()
out = out.drop_vars(SEG_VAR).unify_chunks()
for v in out.data_vars:
    out[v].encoding.clear()
for k, v in out.coords.items():
    v.encoding.clear()
    if v.dtype == object:
        out[k] = v.astype('unicode')
out = out.persist()
out.to_zarr(str(T_FINAL), mode='w', zarr_format=2)

timings['aggregation'] = time.time() - t0
print(f'aggregation: {timings["aggregation"]:.1f}s')

In [ ]:
verify = xr.open_zarr(str(T_FINAL))
print(f'output dims: {dict(verify.sizes)}')
n_nonnull = int(verify.costs.notnull().sum())
n_total = int(verify.costs.size)
pct = 100 * n_nonnull / n_total if n_total > 0 else 0
print(f'non-null: {n_nonnull}/{n_total} ({pct:.1f}%)')
if n_nonnull > 0:
    print('pipeline ran end-to-end.')
else:
    print('output is empty, check errors above.')

## results and scaling

In [ ]:
n_workers_test = max(len(client.scheduler_info()['workers']), 1)

print('timings')
print('=' * 50)
total_test = sum(timings.values())
for step, secs in timings.items():
    pct = secs / total_test * 100 if total_test > 0 else 0
    print(f'  {step:<25} {secs:>8.1f}s  ({pct:>4.0f}%)')
print(f'  {"":25} {"":>8}')
print(f'  {"total":<25} {total_test:>8.1f}s')
print(f'  workers: {n_workers_test}')

In [ ]:
scale_scen_wf = (F_N_SCENARIOS * F_N_WORKFLOWS) / (len(T_SCENARIOS) * len(T_WORKFLOWS))
scale_seg = total_seg_ir / T_N_SEGMENTS
scale_samp = F_N_SAMPLES / T_N_SAMP

parallel_multiplier = scale_seg * scale_samp
slr_multiplier = scale_scen_wf

test_wall_hours = total_test / 3600
test_worker_hours = (total_test * n_workers_test) / 3600

slr_time = timings.get('slr_local_global', 0) + timings.get('slr_vlm', 0) + timings.get('slr_combine_save', 0)
pyciam_time = total_test - slr_time

est_slr_worker_h = (slr_time * slr_multiplier * min(200, F_N_WORKERS)) / 3600
est_pyciam_worker_h = (pyciam_time * n_workers_test * parallel_multiplier) / 3600
est_total_worker_h = est_slr_worker_h + est_pyciam_worker_h

est_wall_800w = est_total_worker_h / F_N_WORKERS
est_wall_200w = est_total_worker_h / 200
est_wall_40w = est_total_worker_h / 40

print('scaling')
print('=' * 50)
print(f'  slr processing scales: {slr_multiplier:.0f}x')
print(f'  pyciam steps scale:    {parallel_multiplier:.0f}x (segments x samples)')
print()
print(f'  test used:             {test_worker_hours:.1f} worker-hours')
print(f'  full run would use:    {est_total_worker_h:.0f} worker-hours')
print(f'  cost multiplier:       {est_total_worker_h / max(test_worker_hours, 0.01):.0f}x the test cost')
print()
print('  wall time depends on cluster size:')
print(f'    40 workers:  {est_wall_40w:.1f} hours')
print(f'    200 workers: {est_wall_200w:.1f} hours')
print(f'    800 workers: {est_wall_800w:.1f} hours')
print()
print('  more workers = faster but same total worker-hours (same cost).')
print('  the tradeoff is speed vs how much capacity you reserve.')

In [ ]:
n_workers = len(client.scheduler_info()['workers'])
total_worker_hours = (total_test * n_workers) / 3600

if client.scheduler_info()['workers']:
    sample_worker = list(client.scheduler_info()['workers'].values())[0]
    memory_gb = sample_worker.get('memory_limit', 0) / 1e9
    n_threads = sample_worker.get('nthreads', 1)
else:
    memory_gb = 0
    n_threads = 1

print('actual cluster usage for this test')
print('=' * 50)
print(f'  workers:       {n_workers}')
print(f'  per worker:    {n_threads} threads, {memory_gb:.1f} gb ram')
print(f'  wall time:     {total_test:.0f}s')
print(f'  worker-hours:  {total_worker_hours:.2f}')

In [ ]:
report = {
    'test': {
        'scenarios': T_SCENARIOS,
        'workflows': T_WORKFLOWS,
        'n_samples': T_N_SAMP,
        'n_segments': T_N_SEGMENTS,
        'total_seg_ir': total_seg_ir,
        'n_workers': n_workers_test,
        'timings_seconds': timings,
        'total_seconds': total_test,
        'worker_hours': test_worker_hours,
    },
    'full_run_estimate': {
        'worker_hours': est_total_worker_h,
        'cost_multiplier': est_total_worker_h / max(test_worker_hours, 0.01),
        'wall_hours_40w': est_wall_40w,
        'wall_hours_200w': est_wall_200w,
        'wall_hours_800w': est_wall_800w,
    }
}
print(json.dumps(report, indent=2, default=str))

In [ ]:
client.close()
cluster.close()
print('done.')